In [2]:
from datasets import load_dataset
import math

In [3]:


ds = load_dataset("avsolatorio/mteb-emotion-avs_triplets")
print(ds)
print(ds["train"][0]) 

df = ds["train"].to_pandas()

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text', 'idx', 'query_idx', 'positive_idx', 'negative_idx'],
        num_rows: 16000
    })
})
{'text': 'i just don t feel i have it in me to get out of bed i can will the dull throbbing of hopelessness to give way and let forth a renewed sensed of hope reflect back on my accomplishments and dig up the inner strength i ve worked so very hard to reestablish', 'label': 0, 'label_text': 'sadness', 'idx': 0, 'query_idx': 0, 'positive_idx': 8206, 'negative_idx': 8560}


In [4]:
word_counts = df['text'].str.split(expand=True).stack().value_counts()

In [5]:
word_counts.head(10)  # Display the top 10 most common words

i          25859
feel       11183
and         9589
to          8972
the         8370
a           6200
that        5112
feeling     5112
of          4990
my          4283
Name: count, dtype: int64

In [6]:
label_counts = df['label'].value_counts()
print(label_counts) 

label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64


In [ ]:
split = ds["train"].train_test_split(test_size=0.2, seed=42)
train_ds, test_ds = split["train"], split["test"]

train_label_counts = dict()
freq_p_class = dict()
for sample in train_ds:
    label = sample["label"]
    train_label_counts[label] = train_label_counts.get(label, 0) + 1
    for word in sample["text"].split():
        if word not in freq_p_class:
            freq_p_class[word] = dict()
        freq_p_class[word][label] = freq_p_class[word].get(label, 0) + 1

v = len(freq_p_class)
total_words = {l: 0 for l in train_label_counts}
for counts in freq_p_class.values():
    for l, c in counts.items():
        total_words[l] += c

In [ ]:
def get_score(text, label):
    score = math.log(train_label_counts[label] / len(train_ds))
    for word in text.split():
        if word not in freq_p_class:
            continue
        score += math.log((freq_p_class[word].get(label, 0) + 1) / (total_words[label] + v))
    return score


def predict(text):
    return max(train_label_counts, key=lambda l: get_score(text, l))

In [9]:
preds = [predict(s["text"]) for s in test_ds]
golds = test_ds["label"]

accuracy = sum(p == g for p, g in zip(preds, golds)) / len(golds)
print(f"accuracy: {accuracy:.4f}")

f1s = []
for l in sorted(train_label_counts):
    tp = sum(p == l and g == l for p, g in zip(preds, golds))
    fp = sum(p == l and g != l for p, g in zip(preds, golds))
    fn = sum(p != l and g == l for p, g in zip(preds, golds))
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    f1s.append(f1)
    print(f"class {l}: f1 {f1:.4f}")
print(f"macro-f1: {sum(f1s) / len(f1s):.4f}")

accuracy: 0.7266
class 0: f1 0.8062
class 1: f1 0.7931
class 2: f1 0.2952
class 3: f1 0.6616
class 4: f1 0.6324
class 5: f1 0.0163
macro-f1: 0.5341
